# 🎓 Capstone Project: Hallucination Detection in Large Language Models (LLMs)
### Domain: NCERT History & Civics Knowledge Base
### Architecture: RAG-Retrieved Evidence + Candidate Answer Comparison via LLM-Based Evaluator

---

## 📌 Proposed Methodology Flow
```
                 [User Input]
              (Question + AI Answer)
                     /    \
                    /      \
                   v        v
              [Question]   [AI-generated answer]
                  |                 |
                  v                 |
         [Retrieve Evidence]        |
          (RAG / Vector DB)         |
                  \                /
                   \              /
                    v            v
            [Evidence + Answer Comparison]
                         |
                         v
               [LLM-Based Evaluator]
       /         /       |       \         \
      v         v        v        v         v
  [Verdict] [Confidence] [Evidence] [Status] [Explanation]
```

### Output Deliverables:
1. **Verdict**: `FAITHFUL / GROUNDED` | `HALLUCINATED` | `PARTIALLY HALLUCINATED`
2. **Confidence**: Evaluator confidence score (e.g., `95%`)
3. **Evidence**: Ground-truth extracted quotes and chapter citations from NCERT
4. **Status**: `Verified / Supported` | `Contradicted` | `Unsupported / Fabricated`
5. **Explanation**: Step-by-step breakdown analyzing each claim against textbook evidence

## Step 1: Install Dependencies
Run the following cell in Google Colab to install the required libraries:

In [ ]:
!pip install -q google-genai chromadb sentence-transformers pypdf gradio

## Step 2: Configure Gemini API Key
Get your free Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey). You can either add it to Colab Secrets (the key icon on the left panel) as `GEMINI_API_KEY` or paste it directly below.

In [ ]:
import os
import getpass

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY', '')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass('Enter your Gemini API Key (from Google AI Studio): ')

os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
print('✅ API Key configured successfully!')

## Step 3: NCERT Dataset Ingestion & Fallback Data Setup
This cell sets up the NCERT textbook dataset. If you have uploaded your PDF files to `/content/data` or `/content`, it will read directly from them. If no PDF is uploaded, it automatically initializes the embedded verified NCERT chapters (*Class 8 History: Our Pasts - III*):

In [ ]:
import os
import pypdf

# Directory for PDFs in Colab
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

# Fallback pre-extracted textbook passages from the 4 uploaded NCERT chapters
NCERT_PRELOADED_CHAPTERS = [
    {
        "source": "NCERT_Ch1_How_When_Where.pdf",
        "page": 1,
        "text": "In 1782, James Rennel produced the first map of Hindustan. Robert Clive asked Rennel to produce maps of Hindustan. Rennel saw preparation of maps as essential to the process of British domination in India."
    },
    {
        "source": "NCERT_Ch1_How_When_Where.pdf",
        "page": 3,
        "text": "Warren Hastings became the first Governor-General of India in 1773. In 1817, James Mill, a Scottish economist and political philosopher, published a massive three-volume work, A History of British India. In this he divided Indian history into three periods - Hindu, Muslim and British. This periodisation came to be widely accepted, though later historians moved away from it and classified periods as ancient, medieval, and modern or colonial."
    },
    {
        "source": "NCERT_Ch1_How_When_Where.pdf",
        "page": 5,
        "text": "Official records of the British administration are an important source. The British set up record rooms attached to all administrative institutions like tahsildar offices, collectorates, and provincial secretariats. The National Archives of India came up in the 1920s in New Delhi."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 10,
        "text": "In 1600, the East India Company acquired a charter from the ruler of England, Queen Elizabeth I, granting it the sole right to trade with the East. This meant that no other trading group in England could compete with the East India Company. Vasco da Gama, a Portuguese explorer, had discovered the sea route to India in 1498."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 11,
        "text": "The first English factory was set up on the banks of the river Hugli in 1651. In 1696 the Company began building a fort around the settlement and obtained zamindari rights over three villages, including Kalikata (Calcutta). Mughal emperor Aurangzeb issued a farman granting the Company the right to trade duty-free."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 12,
        "text": "When Alivardi Khan died in 1756, Sirajuddaulah became the Nawab of Bengal. In 1757, Robert Clive led the Company army against Sirajuddaulah at the Battle of Plassey. The main reason for the defeat of the Nawab was that the forces led by Mir Jafar, one of Sirajuddaulah commanders, never fought the battle because Clive promised to make him nawab."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 14,
        "text": "After Plassey, Mir Jafar was made nawab. Later Mir Qasim was installed, but after his complaint he was defeated in the Battle of Buxar in 1764. In 1765 the Mughal emperor appointed the East India Company as the Diwan of the provinces of Bengal, allowing the Company to use Bengal revenues to buy goods and maintain troops."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 16,
        "text": "Under the Subsidiary Alliance system, Indian rulers were not allowed to have their independent armed forces. They were protected by the Company and had to pay for subsidiary forces. If they failed to pay, part of their territory was seized. Awadh and Hyderabad were forced to cede territories on these grounds."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 17,
        "text": "Tipu Sultan, the Tiger of Mysore, ruled Mysore from 1782 to 1799 following his father Haidar Ali. Tipu stopped export of sandalwood, pepper, and cardamom and modernised his army with French help. After four Anglo-Mysore wars, Tipu was killed in 1799 defending his capital Seringapatam."
    },
    {
        "source": "NCERT_Ch2_From_Trade_to_Territory.pdf",
        "page": 19,
        "text": "Lord Dalhousie, Governor-General from 1848 to 1856, devised the Doctrine of Lapse. It declared that if an Indian ruler died without a male heir, his kingdom would lapse and become Company territory. Kingdoms annexed under this doctrine included Satara (1848), Sambalpur (1850), Udaipur (1852), Nagpur (1853), and Jhansi (1854). Awadh was annexed in 1856 on charges of misgovernment."
    },
    {
        "source": "NCERT_Ch4_Tribals_Dikus.pdf",
        "page": 39,
        "text": "In 1895, Birsa was seen roaming the forests of Chottanagpur in Jharkhand. Birsa was born in a family of Mundas, and his followers included Santhals and Oraons. Birsa declared that he had come to free his people from the slavery of dikus (outsiders) like moneylenders, traders, and the British."
    },
    {
        "source": "NCERT_Ch4_Tribals_Dikus.pdf",
        "page": 40,
        "text": "Shifting cultivation (jhum cultivation) was practised by cutting treetops, burning vegetation to spread potash-rich ash, and broadcasting seeds. Khonds of Orissa were hunters and gatherers who extracted oil from sal and mahua seeds. Baigas of central India saw themselves as people of the forest and considered it below their dignity to become labourers."
    },
    {
        "source": "NCERT_Ch4_Tribals_Dikus.pdf",
        "page": 45,
        "text": "Under British rule, forests were declared state property and classified as Reserved Forests. Tribals were not allowed to practise jhum or gather produce freely. Birsa Munda was arrested in 1895 and released in 1897. He died of cholera in 1900. His movement forced the British to introduce laws protecting tribal land."
    },
    {
        "source": "NCERT_Ch6_Colonialism_City.pdf",
        "page": 66,
        "text": "Under British rule, older cities declined while Calcutta, Bombay, and Madras grew as Presidency cities. Ports like Machlipatnam, Surat, and Seringapatam suffered de-urbanisation. By the early twentieth century, only 11 percent of Indians lived in cities."
    },
    {
        "source": "NCERT_Ch6_Colonialism_City.pdf",
        "page": 67,
        "text": "Shahjahanabad was begun by Shah Jahan in 1639, consisting of the Red Fort and the walled city with 14 gates, Chandni Chowk, and the Jama Masjid. The period from 1830 to 1857 was known as the Delhi Renaissance."
    },
    {
        "source": "NCERT_Ch6_Colonialism_City.pdf",
        "page": 71,
        "text": "In 1911, King George V was crowned in England, and at the Delhi Durbar the shift of capital from Calcutta to Delhi was announced. Architects Edward Lutyens and Herbert Baker designed New Delhi on Raisina Hill, including the Viceroy Palace (Rashtrapati Bhavan)."
    }
]
print(f"Loaded {len(NCERT_PRELOADED_CHAPTERS)} verified NCERT historical passages into knowledge base!")

## Step 4: Vector Database (ChromaDB + Sentence Transformers)
Initialize ChromaDB and create vector embeddings using `all-MiniLM-L6-v2` (runs 100% free on Colab CPU/T4 GPU):

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="ncert_history", embedding_function=embedding_fn)

# Index any PDFs in DATA_DIR if present
indexed_docs = []
indexed_metas = []
indexed_ids = []

pdf_found = False
for fname in os.listdir(DATA_DIR):
    if fname.lower().endswith('.pdf'):
        pdf_found = True
        reader = pypdf.PdfReader(os.path.join(DATA_DIR, fname))
        for idx, page in enumerate(reader.pages):
            text = page.extract_text()
            if text and text.strip():
                indexed_docs.append(text.strip())
                indexed_metas.append({"source": fname, "page": idx + 1})
                indexed_ids.append(f"{fname}_p{idx+1}")

if not pdf_found:
    print("ℹ️ No custom PDFs uploaded to /data. Indexing built-in NCERT chapters...")
    for i, item in enumerate(NCERT_PRELOADED_CHAPTERS):
        indexed_docs.append(item["text"])
        indexed_metas.append({"source": item["source"], "page": item["page"]})
        indexed_ids.append(f"ncert_doc_{i}")

collection.upsert(documents=indexed_docs, metadatas=indexed_metas, ids=indexed_ids)
print(f"✅ Successfully indexed {collection.count()} chunks into Chroma Vector DB!")

## Step 5: Implement RAG Retriever & LLM Evaluator
This implements the core capstone methodology:
- **RAG Retriever**: Queries vector DB for relevant NCERT evidence.
- **LLM Evaluator**: Deeply compares the evidence and the AI answer to output:
  - `verdict`
  - `confidence`
  - `evidence`
  - `status`
  - `explanation`

In [ ]:
import time
import re
import json
from google import genai

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

# Comprehensive fallback list covering all stable Gemini tiers
MODELS_TO_TRY = [
    'gemini-2.5-flash',
    'gemini-2.0-flash',
    'gemini-1.5-flash',
    'gemini-2.5-pro',
    'gemini-1.5-pro'
]

def call_gemini(contents):
    last_err = None
    for m in MODELS_TO_TRY:
        for attempt in range(2):
            try:
                resp = client.models.generate_content(model=m, contents=contents)
                return resp.text.strip()
            except Exception as e:
                last_err = e
                err_str = str(e).upper()
                # If model is unavailable (503), rate-limited (429), or deprecated (404), fall back to next model
                if any(k in err_str for k in ['503', 'UNAVAILABLE', 'HIGH DEMAND', '429', 'RESOURCE_EXHAUSTED', '404', 'NOT_FOUND']):
                    time.sleep(1.5)
                    continue
                break
    raise RuntimeError(f'Gemini API request failed across all models. Details: {last_err}')

def retrieve_evidence(query: str, top_k: int = 3):
    """Fetches top-k relevant NCERT passages."""
    results = collection.query(query_texts=[query], n_results=min(top_k, collection.count()))
    evidences = []
    if results and results["documents"] and results["documents"][0]:
        for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
            evidences.append({
                "content": doc,
                "source": meta.get("source", "NCERT"),
                "page": meta.get("page", 1)
            })
    return evidences

def generate_candidate_answer(question: str) -> str:
    """Generates a candidate answer from an LLM."""
    prompt = f"Answer this student history question concisely in 2-3 sentences based on Indian history: {question}"
    return call_gemini(prompt)

def evaluate_hallucination(question: str, candidate_answer: str, retrieved_evidences: list) -> dict:
    """Evidence + Answer Comparison via LLM Evaluator."""
    evidence_text = "\n\n".join([
        f"[Source: {e['source']}, Page: {e['page']}]\n{e['content']}"
        for e in retrieved_evidences
    ])

    eval_prompt = f"""You are an expert AI Hallucination Evaluator for educational NCERT History content.
Your task is to compare the [AI GENERATED ANSWER] against the [RETRIEVED NCERT EVIDENCE] for the given [QUESTION].

[QUESTION]:
{question}

[AI GENERATED ANSWER]:
{candidate_answer}

[RETRIEVED NCERT EVIDENCE]:
{evidence_text}

Determine:
1. VERDICT: "FAITHFUL / GROUNDED" or "HALLUCINATED" or "PARTIALLY HALLUCINATED"
2. CONFIDENCE: Percentage score e.g. "95%"
3. STATUS: "Verified / Supported" or "Contradicted" or "Unsupported / Fabricated" or "Partially Supported"
4. EVIDENCE: Exact quotes from the NCERT evidence supporting or refuting the claim.
5. EXPLANATION: Step-by-step reasoning explaining why it is factual or hallucinated.

Return ONLY a valid JSON object (no markdown code blocks, just raw JSON):
{{
  "verdict": "VERDICT_VALUE",
  "confidence": "CONFIDENCE_VALUE",
  "status": "STATUS_VALUE",
  "evidence": "EXACT_QUOTES_WITH_CITATION",
  "explanation": "STEP_BY_STEP_REASONING"
}}"""

    raw = call_gemini(eval_prompt)
    cleaned = re.sub(r"^```json\s*", "", raw, flags=re.MULTILINE)
    cleaned = re.sub(r"^```\s*", "", cleaned, flags=re.MULTILINE).strip()
    try:
        return json.loads(cleaned)
    except Exception:
        return {
            "verdict": "ERROR",
            "confidence": "0%",
            "status": "Failed",
            "evidence": evidence_text[:250],
            "explanation": raw
        }

print("✅ RAG Retriever and LLM Evaluator initialized successfully!")

## Step 6: Capstone Test Bench (Benchmark Deliberate Hallucinations vs Grounded Answers)
Let's test both a deliberate hallucination and a verified ground-truth answer:

In [ ]:
# TEST CASE 1: DELIBERATE HALLUCINATION
q1 = "When was the Battle of Plassey fought and who led the British forces?"
fake_answer = "The Battle of Plassey was fought in 1857. The British army was led by Warren Hastings who defeated Tipu Sultan with the help of the French navy."

ev1 = retrieve_evidence(q1, top_k=2)
result1 = evaluate_hallucination(q1, fake_answer, ev1)

print("========== TEST 1: DELIBERATE HALLUCINATION ==========")
print(f"Question: {q1}")
print(f"Candidate Answer: {fake_answer}")
print(f"VERDICT: {result1['verdict']}")
print(f"CONFIDENCE: {result1['confidence']}")
print(f"STATUS: {result1['status']}")
print(f"EVIDENCE: {result1['evidence']}")
print(f"EXPLANATION: {result1['explanation']}\n")

# TEST CASE 2: GROUNDED / FAITHFUL ANSWER
q2 = "Who devised the Doctrine of Lapse and name some kingdoms annexed under it?"
grounded_answer = "Lord Dalhousie devised the Doctrine of Lapse. Under this doctrine, kingdoms like Satara, Sambalpur, Udaipur, Nagpur, and Jhansi were annexed."

ev2 = retrieve_evidence(q2, top_k=2)
result2 = evaluate_hallucination(q2, grounded_answer, ev2)

print("========== TEST 2: FAITHFUL / GROUNDED ANSWER ==========")
print(f"Question: {q2}")
print(f"Candidate Answer: {grounded_answer}")
print(f"VERDICT: {result2['verdict']}")
print(f"CONFIDENCE: {result2['confidence']}")
print(f"STATUS: {result2['status']}")
print(f"EVIDENCE: {result2['evidence']}")
print(f"EXPLANATION: {result2['explanation']}")

## Step 7: Launch Interactive Web Interface (Gradio)
This launches the web UI directly inside Colab and provides a public `.gradio.live` link so anyone can test it from their phone or browser.

In [ ]:
import gradio as gr

def badge(v):
    c = "#0d9488" if "FAITHFUL" in v.upper() or "GROUNDED" in v.upper() else ("#d97706" if "PARTIAL" in v.upper() else "#dc2626")
    return f'<div style="background-color:{c}; color:white; padding:12px; border-radius:8px; font-weight:bold; font-size:18px; text-align:center;">{v}</div>'

def run_full_pipeline(question, custom_answer):
    if not question.strip():
        return "", badge("EMPTY INPUT"), "0%", "None", "", ""
    ans = custom_answer.strip() if custom_answer.strip() else generate_candidate_answer(question)
    ev = retrieve_evidence(question, top_k=3)
    res = evaluate_hallucination(question, ans, ev)
    return ans, badge(res["verdict"]), res["confidence"], res["status"], res["evidence"], res["explanation"]

with gr.Blocks(title="Hallucination Detection in LLMs", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🛡️ Capstone: Hallucination Detection in LLMs\n### NCERT History & Civics Ground Truth Verification")
    with gr.Row():
        with gr.Column(scale=5):
            inp_q = gr.Textbox(label="1. Question", placeholder="e.g. When was the Battle of Plassey fought?", lines=2)
            inp_ans = gr.Textbox(label="2. Candidate AI Answer (Leave empty to let AI generate, or type your own to test hallucinations)", lines=3)
            btn = gr.Button("🚀 Run Hallucination Detection", variant="primary")
        with gr.Column(scale=5):
            out_badge = gr.HTML(badge("AWAITING INPUT"))
            with gr.Row():
                out_conf = gr.Textbox(label="Confidence", interactive=False)
                out_stat = gr.Textbox(label="Status", interactive=False)
            out_ev = gr.Textbox(label="Retrieved NCERT Evidence", lines=4, interactive=False)
            out_exp = gr.Textbox(label="Evaluator Explanation", lines=4, interactive=False)
    btn.click(run_full_pipeline, inputs=[inp_q, inp_ans], outputs=[inp_ans, out_badge, out_conf, out_stat, out_ev, out_exp])

demo.launch(share=True, debug=False)